# Qwen2.5-0.5B × DPO 偏好對齊（TRL + LoRA，Colab T4/L4）

用 TRL 的 `DPOTrainer` 對 [`Qwen/Qwen2.5-0.5B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) 做 DPO 偏好對齊，資料集為 [`HuggingFaceH4/ultrafeedback_binarized`](https://huggingface.co/datasets/HuggingFaceH4/ultrafeedback_binarized)。

**DPO 一句話**：不訓練 reward model、不跑強化學習迴圈，直接用「同一個 prompt 的較好／較差回答」成對訓練；隱式獎勵是 policy 與 reference 模型的 log 機率比值，`beta` 控制可以偏離 reference 的程度（大＝保守、小＝激進）。用了 LoRA 之後連 reference 模型都不用另外載——關掉 adapter 的 base 就是 reference。

**執行前準備**：

1. `執行階段 → 變更執行階段類型 → T4 或 L4 GPU`（程式會自動偵測 GPU 選 fp16／bf16）。
2. 左側鑰匙圖示（Secrets）新增 `HF_TOKEN`：**write 權限**的 Hugging Face token，並開啟「筆記本存取權」。⚠️ read-only token 要到最後推送模型那步才會失敗——訓練完一小時才發現就太遲了。
3. 建議先**勾選**下面參數 cell 右側的 `SMOKE_TEST` 核取方塊、全部執行一次（約 3–5 分鐘）驗證整條流程；通過後**取消勾選**，用「執行階段 → 重新啟動工作階段並全部執行」正式訓練（6000 對、T4 約 40–70 分鐘、L4 約 20–35 分鐘）。


In [ ]:
# ===== 參數（整本 notebook 只需要改這裡）=====
MODEL   = "Qwen/Qwen2.5-0.5B-Instruct"
DATASET = "HuggingFaceH4/ultrafeedback_binarized"
SPLIT   = "train_prefs"
GH_REPO = "tun0000/qwen-dpo-alignment"   # 共用格式化模組 scripts/dpo_formatting.py 的來源 repo

N_PAIRS           = 6000     # 從 train_prefs 取樣的偏好對數
BETA              = 0.1      # DPO 的 KL 懲罰係數（越大越緊貼 reference）
LEARNING_RATE     = 5e-6
LORA_R            = 16
LORA_ALPHA        = 32       # 慣例：2 * r
MAX_LENGTH        = 1024     # prompt+completion 合併 token 上限（TRL v1.x 唯一的長度參數）
MAX_PROMPT_TOKENS = 512      # 自家資料過濾用（DPOConfig 已無 max_prompt_length）
SEED              = 42

HUB_MODEL_NAME = "qwen2.5-0.5b-dpo-ultrafeedback"
OUTPUT_DIR     = "dpo-output"

# SMOKE_TEST 勾選（True）＝只用 100 對、跑 20 steps（約 3–5 分鐘）驗證整條流程、不推送；
# 取消勾選（False）＝正式訓練。用 cell 右側的核取方塊切換即可，不必改程式碼。
SMOKE_TEST = False  # @param {type:"boolean"}
PUSH_TO_HUB = True

if SMOKE_TEST:
    N_PAIRS = 100
    MAX_STEPS = 20
    LOGGING_STEPS = 1
    GEN_MAX_NEW_TOKENS = 64
    PUSH_TO_HUB = False   # 冒煙測試不推送
else:
    MAX_STEPS = -1        # -1 = 跑滿 num_train_epochs
    LOGGING_STEPS = 10
    GEN_MAX_NEW_TOKENS = 256

print(f"SMOKE_TEST={SMOKE_TEST}  N_PAIRS={N_PAIRS}  MAX_STEPS={MAX_STEPS}  PUSH_TO_HUB={PUSH_TO_HUB}")


In [ ]:
# 安裝依賴（若下一個 cell 出現 ImportError：執行階段 → 重新啟動工作階段，再從頭執行）
%pip install -q -U "trl>=1.0,<2" peft transformers datasets accelerate bitsandbytes
# Colab 預裝的 torchao 0.10 與新版 peft 不相容（peft 要 >=0.16，套件不存在則會乾淨跳過），
# 本專案不做量化、用不到 torchao，直接移除最保險
%pip uninstall -q -y torchao

from importlib.metadata import version
for pkg in ["trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes", "torch"]:
    print(f"{pkg:>13} {version(pkg)}")


In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))  # 需要 write 權限（最後要推送模型）

import torch
from transformers import set_seed

assert torch.cuda.is_available(), "請切換 GPU 執行階段：執行階段 → 變更執行階段類型 → T4 或 L4"
USE_BF16 = torch.cuda.is_bf16_supported()   # T4(sm75)→False、L4(sm89)→True
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"GPU: {torch.cuda.get_device_name(0)} | 訓練 dtype: {DTYPE}")

set_seed(SEED)


## 1. 載入資料集，先看清楚原始格式

`ultrafeedback_binarized` 的 `chosen` / `rejected` **不是純文字，而是 message list**
（`[{"role": "user", ...}, {"role": "assistant", ...}]`），兩者共享同一個 user prompt，
只有最後的 assistant 回覆不同。下面先印出欄位與一筆完整樣本確認，
再轉成 `DPOTrainer` 需要的「標準格式」：`prompt` / `chosen` / `rejected` 三個純字串，
並套上 Qwen 的 chat template。


In [ ]:
import json
from datasets import load_dataset

raw_ds = load_dataset(DATASET, split=SPLIT)
print("欄位:", raw_ds.column_names)
print("筆數:", len(raw_ds))
print()
print(json.dumps(raw_ds[0], indent=2, ensure_ascii=False))  # 完整印出 1 筆樣本


In [ ]:
# 下載與本機 scripts/preview_pairs.py 共用的格式化模組（單一正本，保證兩邊邏輯一致）
import importlib
import urllib.request

RAW_URL = f"https://raw.githubusercontent.com/{GH_REPO}/main/scripts/dpo_formatting.py"
try:
    urllib.request.urlretrieve(RAW_URL, "dpo_formatting.py")
except Exception as e:
    raise RuntimeError(
        f"無法下載 {RAW_URL} — 請確認參數 GH_REPO 正確、且 repo 已 push 到 GitHub 的 main 分支"
    ) from e

import dpo_formatting
importlib.reload(dpo_formatting)  # 改過 GH_REPO 重跑時確保拿到新版
from dpo_formatting import to_dpo_format

print("已載入共用格式化模組:", dpo_formatting.__file__)


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
print("eos:", repr(tokenizer.eos_token), "| pad:", repr(tokenizer.pad_token))

# 過濾 1：分數要有明確偏好、內容非空且 chosen != rejected
def keep_pair(ex):
    if ex["score_chosen"] <= ex["score_rejected"]:
        return False
    c = ex["chosen"][-1]["content"].strip()
    r = ex["rejected"][-1]["content"].strip()
    return bool(c) and bool(r) and c != r

ds = raw_ds.filter(keep_pair, num_proc=2)
print(f"分數/內容過濾: {len(raw_ds)} -> {len(ds)}")

# 轉成標準格式。remove_columns 是關鍵：若殘留 message-list 欄位，
# TRL 會誤判為 conversational 格式、對已套好 template 的字串再套一次 template
ds = ds.map(
    to_dpo_format,
    fn_kwargs={"tokenizer": tokenizer},
    remove_columns=raw_ds.column_names,
    num_proc=2,
)

# 過濾 2：token 長度預算。TRL v1.x 的 DPOConfig 沒有 max_prompt_length，
# collator 會對 prompt+completion 合併截斷（keep_start）——截到 completion 尾端
# 會毀掉 EOS 訊號，所以事前把過長樣本整筆濾掉，保證截斷永不觸發
def within_budget(ex):
    p = len(tokenizer(ex["prompt"], add_special_tokens=False)["input_ids"])
    c = len(tokenizer(ex["chosen"], add_special_tokens=False)["input_ids"])
    r = len(tokenizer(ex["rejected"], add_special_tokens=False)["input_ids"])
    return p <= MAX_PROMPT_TOKENS and p + c <= MAX_LENGTH - 1 and p + r <= MAX_LENGTH - 1

ds = ds.filter(within_budget, num_proc=2)
print(f"長度過濾後: {len(ds)}")

train_ds = ds.shuffle(seed=SEED).select(range(min(N_PAIRS, len(ds))))
print(train_ds)
print()
print("prompt 開頭:", repr(train_ds[0]["prompt"][:200]))
print()
print("chosen 結尾:", repr(train_ds[0]["chosen"][-80:]))  # 應恰以 <|im_end|> 收尾（不重複、無殘留換行）


## 2. LoRA + DPO 訓練

- **為什麼不用載 reference model**：傳給 `DPOTrainer` 一個 `peft_config` 之後，TRL 會把 base 模型包成 PeftModel；算 reference log-prob 時直接**關掉 adapter** 再 forward 一次——省一份模型的記憶體。
- **dtype**：T4 不支援 bf16，前面已自動偵測（T4→fp16、L4→bf16）。LoRA 權重本身會保持 fp32，所以 fp16 下 GradScaler 也能正常運作。
- **預期指標**：`rewards/accuracies`（隱式獎勵排序正確率）應從 ~0.5 爬向 0.6–0.75；`rewards/margins` 持續拉大；loss 從 ~0.693（ln 2）下降。
- 若 OOM：把 `per_device_train_batch_size` 降為 2、`gradient_accumulation_steps` 升為 8（等效 batch 不變）。


In [ ]:
import math

from peft import LoraConfig
from transformers import AutoModelForCausalLM
from trl import DPOConfig, DPOTrainer

model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE, device_map="auto")
model.config.use_cache = False  # gradient checkpointing 與 KV cache 不相容

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

PER_DEVICE_BS = 4   # OOM 時降為 2，並把 GRAD_ACCUM 升為 8（等效 batch 不變）
GRAD_ACCUM = 4      # 等效 batch = PER_DEVICE_BS * GRAD_ACCUM = 16

# transformers v5 已棄用 warmup_ratio（v5.2 將移除），自行換算成 warmup 步數
total_steps = MAX_STEPS if MAX_STEPS > 0 else math.ceil(len(train_ds) / (PER_DEVICE_BS * GRAD_ACCUM))
WARMUP_STEPS = max(1, int(0.1 * total_steps))

training_args = DPOConfig(
    output_dir=OUTPUT_DIR,
    beta=BETA,
    loss_type="sigmoid",                 # 原始 DPO 論文的 loss
    max_length=MAX_LENGTH,
    per_device_train_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=1,                  # DPO 很容易過擬合，1 epoch 是常規
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    fp16=(DTYPE == torch.float16),
    bf16=(DTYPE == torch.bfloat16),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=LOGGING_STEPS,
    save_strategy="no",                  # 不落地 checkpoint，最後直接推送
    report_to="none",
    seed=SEED,
    dataset_num_proc=2,
)

trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    processing_class=tokenizer,
    peft_config=peft_config,  # 有 peft_config 就不需 ref_model
)
trainer.train()


In [ ]:
import matplotlib.pyplot as plt

hist = trainer.state.log_history
steps   = [h["step"] for h in hist if "rewards/accuracies" in h]
acc     = [h["rewards/accuracies"] for h in hist if "rewards/accuracies" in h]
margins = [h["rewards/margins"] for h in hist if "rewards/margins" in h]
lsteps  = [h["step"] for h in hist if "loss" in h]
loss    = [h["loss"] for h in hist if "loss" in h]

def smooth(xs, w=5):
    out = []
    for i in range(len(xs)):
        window = xs[max(0, i - w + 1): i + 1]
        out.append(sum(window) / len(window))
    return out

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(lsteps, loss, alpha=0.35)
axes[0].plot(lsteps, smooth(loss), lw=2)
axes[0].set_title("DPO loss")
axes[1].plot(steps, acc, alpha=0.35)
axes[1].plot(steps, smooth(acc), lw=2)
axes[1].axhline(0.5, ls="--", c="gray", lw=1)
axes[1].set_title("rewards/accuracies")
axes[2].plot(steps, margins, alpha=0.35)
axes[2].plot(steps, smooth(margins), lw=2)
axes[2].set_title("rewards/margins")
for ax in axes:
    ax.set_xlabel("step")
fig.suptitle(f"Qwen2.5-0.5B DPO on UltraFeedback (beta={BETA}, lr={LEARNING_RATE}, LoRA r={LORA_R})")
fig.tight_layout()
fig.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

FINAL_ACC = smooth(acc)[-1] if acc else float("nan")
print(f"最終 rewards/accuracies（滾動平均）: {FINAL_ACC:.3f}")


## 3. 訓練前 vs 訓練後：並排對照

同一份權重就能對照：`policy.disable_adapter()` 是 PEFT 的 context manager，
進入時暫時關閉 LoRA adapter（＝原始模型），離開後 adapter 恢復（＝DPO 後模型），
不需要重新載入第二個模型。用 greedy decoding 讓結果可重現。


In [ ]:
import gc
import html as html_lib
from IPython.display import HTML, display

# 釋放訓練狀態（optimizer 的 Adam 狀態佔不少 VRAM），並恢復推論設定
trainer.optimizer = None
gc.collect()
torch.cuda.empty_cache()

policy = trainer.model  # PeftModel（掛著 LoRA adapter）
policy.eval()
policy.gradient_checkpointing_disable()
policy.config.use_cache = True  # 生成需要 KV cache

EVAL_PROMPTS = [
    "What are the health benefits of intermittent fasting? Are there any risks I should know about?",
    "Explain the difference between supervised and unsupervised learning to a high school student.",
    "I have chicken, rice, and broccoli. Give me a simple 30-minute dinner recipe.",
    "Write a short professional email asking my manager for two days off next week.",
    "What would happen if the Moon suddenly disappeared? Walk me through the consequences.",
]

def generate(model, prompt):
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=GEN_MAX_NEW_TOKENS,
            do_sample=False, temperature=None, top_p=None, top_k=None,  # greedy、可重現
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

comparison_results = []
for p in EVAL_PROMPTS:
    with policy.disable_adapter():      # 關閉 LoRA -> 原始模型
        base_out = generate(policy, p)
    dpo_out = generate(policy, p)       # context 外 adapter 生效 -> DPO 後模型
    comparison_results.append({"prompt": p, "base": base_out, "dpo": dpo_out})
    print("完成:", p[:60])

def td(content, bold=False):
    inner = html_lib.escape(content)
    if bold:
        inner = f"<b>{inner}</b>"
    return ("<td style='vertical-align:top;padding:8px;border:1px solid #999'>"
            f"<pre style='white-space:pre-wrap;margin:0;font-family:inherit'>{inner}</pre></td>")

rows = "".join(
    "<tr>" + td(r["prompt"], bold=True) + td(r["base"]) + td(r["dpo"]) + "</tr>"
    for r in comparison_results
)
display(HTML(
    "<table style='table-layout:fixed;width:100%;border-collapse:collapse'>"
    "<tr><th style='width:22%'>Prompt</th><th style='width:39%'>原始模型</th>"
    "<th style='width:39%'>DPO 後模型</th></tr>" + rows + "</table>"
))


## 4. 合併 LoRA 並推送到 Hugging Face Hub

推送 **merged 模型**（`merge_and_unload()` 把 LoRA 併回 base 權重、維持半精度），
下游使用者直接 `from_pretrained` 即可，不需要安裝 PEFT。model card 會自動生成：
方法說明、超參數、訓練曲線圖、before/after 對照。

⚠️ merge 之後就無法再 `disable_adapter()` 對照了，所以此 cell 必須在上面的對照 cell 之後執行。


In [ ]:
from huggingface_hub import HfApi

api = HfApi()
HF_USERNAME = api.whoami()["name"]
REPO_ID = f"{HF_USERNAME}/{HUB_MODEL_NAME}"

if not PUSH_TO_HUB:
    print("PUSH_TO_HUB=False（SMOKE_TEST 模式）— 跳過推送。")
else:
    merged = trainer.model.merge_and_unload()
    merged.config.use_cache = True
    print("merged dtype:", next(merged.parameters()).dtype)  # 應為 float16 / bfloat16

    merged.push_to_hub(REPO_ID)
    tokenizer.push_to_hub(REPO_ID)

    examples_md = "\n".join(
        f"**Prompt:** {r['prompt']}\n\n"
        f"<details><summary>原始模型</summary>\n\n{r['base']}\n\n</details>\n"
        f"<details><summary>DPO 後模型</summary>\n\n{r['dpo']}\n\n</details>\n"
        for r in comparison_results
    )
    dtype_name = str(DTYPE).split(".")[-1]

    model_card = f"""---
license: apache-2.0
base_model: {MODEL}
datasets:
- {DATASET}
library_name: transformers
pipeline_tag: text-generation
tags:
- dpo
- trl
- peft
- qwen2.5
- preference-alignment
---

# {HUB_MODEL_NAME}

`{MODEL}` 以 **DPO（Direct Preference Optimization）** 在
[{DATASET}](https://huggingface.co/datasets/{DATASET})（`{SPLIT}` split）的
{len(train_ds)} 組偏好資料對上做偏好對齊。

## 方法：DPO 是什麼？

DPO 直接用「同一個 prompt 的較好回答（chosen）與較差回答（rejected）」成對訓練，
不需要先訓練獨立的 reward model、也不需要 PPO 的強化學習迴圈。它把隱式獎勵定義為
policy 與 reference 模型的 log 機率比值：r(x, y) = beta * log(pi(y|x) / pi_ref(y|x))，
訓練目標是把 chosen 與 rejected 的隱式獎勵差距拉大。

**beta = {BETA}**：控制 policy 可以偏離 reference 的程度——beta 越大越保守
（緊貼 reference），越小越激進（對齊訊號強、但容易偏離原模型導致輸出劣化）。

本 repo 存放的是 **merged 模型**：LoRA adapter（r={LORA_R}）已用 `merge_and_unload()`
併回 base 權重（{dtype_name}），推論時**不需要 PEFT**，直接 `from_pretrained` 即可。

## 超參數

| beta | learning rate | LoRA r / alpha | 等效 batch | epochs | max_length | dtype |
|------|---------------|----------------|-----------|--------|------------|-------|
| {BETA} | {LEARNING_RATE} | {LORA_R} / {LORA_ALPHA} | 16 | 1 | {MAX_LENGTH} | {dtype_name} |

訓練框架：TRL `DPOTrainer`（loss_type=sigmoid）。資料前處理：套 Qwen chat template
轉為標準格式、過濾無明確偏好（score 平手／反轉）與過長樣本。

## 訓練曲線

最終 rewards/accuracies（滾動平均）：**{FINAL_ACC:.3f}**

![training curves](training_curves.png)

## Before / After 對照（greedy decoding）

{examples_md}

## 如何使用

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("{REPO_ID}", dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("{REPO_ID}")
```

由 [{GH_REPO}](https://github.com/{GH_REPO}) 的 `dpo_colab.ipynb` 訓練產生。
"""

    with open("MODEL_CARD.md", "w", encoding="utf-8") as f:
        f.write(model_card)
    api.upload_file(path_or_fileobj="MODEL_CARD.md", path_in_repo="README.md", repo_id=REPO_ID)
    api.upload_file(path_or_fileobj="training_curves.png", path_in_repo="training_curves.png", repo_id=REPO_ID)
    print(f"完成: https://huggingface.co/{REPO_ID}")


## 完成！

- 模型連結見上一個 cell 的輸出（`https://huggingface.co/<你的帳號>/qwen2.5-0.5b-dpo-ultrafeedback`）。
- 之後載入不需要 PEFT：

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(
    "<你的帳號>/qwen2.5-0.5b-dpo-ultrafeedback", dtype="auto", device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("<你的帳號>/qwen2.5-0.5b-dpo-ultrafeedback")
```

**延伸實驗**：beta 掃描（0.05 / 0.1 / 0.3）、`loss_type="ipo"`、加大 `N_PAIRS` 或 LoRA r、
用 `test_prefs` split 做正式的 win-rate 評測。
